<a href="https://colab.research.google.com/github/J-Cos/EuropeanRewildingProgress/blob/main/NDVI_Trend_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/J-Cos/EuropeanRewildingProgress/blob/main/NDVI_Trend_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NDVI Trend Analysis: Mann-Kendall Tests & Sen's Slope
## Overview
This notebook computes annual NDVI metrics (INDVI, minNDVI, maxNDVI) from MODIS data
and applies **pixel-wise Mann-Kendall trend tests** across the full MODIS period (2000–present).

Two resolutions are processed in parallel:
- **250 m** (MOD13Q1, 16-day) — QA-filtered and Garonna-smoothed
- **1000 m** (MOD13A3, monthly) — raw scaled NDVI (Williams et al. sensitivity test)

**Outputs per metric (at each resolution):**
- **Mann-Kendall S** — monotonic trend strength and direction
- **Kendall variance** — with tie correction (Sen 1968, eq. 2.6)
- **Z-score** — continuity-corrected standard normal statistic
- **p-value** — two-sided significance
- **Sen's Slope** — robust median rate of change (NDVI units / year)

## Data Sources
| Product | Resolution | Temporal | ID |
|---------|-----------|----------|-----|
| MOD13Q1 | 250 m | 16-day | `MODIS/061/MOD13Q1` |
| MOD13A3 (sensitivity) | 1000 m | Monthly | `MODIS/061/MOD13A3` |
| Dynamic World V1 | 10 m | — | `GOOGLE/DYNAMICWORLD/V1` |

## Preprocessing
1. **250 m**: QA filtering & Garonna et al. (2009) temporal smoothing (as in `MODIS_Demo.ipynb`)
2. **1000 m**: Raw NDVI scaling only (no QA mask, no smoothing)
3. Dynamic World land-cover masking: pixels with >75% combined probability of
   water, crops, built, bare, or snow/ice are excluded

## References
- Garonna et al. (2009) — temporal smoothing
- Schulte to Bühne et al. (2022) — rewilding metric framework
- Williams et al. (2024) — MODIS NDVI trend analysis
- Sen (1968) — variance tie correction (eq. 2.6)

## NOTE
Agentic programming was used. All logic is defined in functions; all functions are unit tested.


## 1. Setup & Control Flags

In [1]:
import ee

# ── Site Selection ─────────────────────────────────────
SITES_ASSET = 'users/AidanHoutenUCL/allsites'

# ── Dynamic World Masking ──────────────────────────────
DW_START          = '2022-04-01'
DW_END            = '2022-09-30'
DW_PROB_THRESHOLD = 0.75

# ── Time Range ─────────────────────────────────────────
END_YEAR = 2025

# ── Metrics to trend-test ──────────────────────────────
METRICS = ['INDVI', 'minNDVI', 'maxNDVI']

# ── EE Init ────────────────────────────────────────────
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')


## 2. Function Definitions

In [2]:
# ═══════════════════════════════════════════════════════
#  PREPROCESSING FUNCTIONS (from MODIS_Demo.ipynb)
# ═══════════════════════════════════════════════════════

def prep_modis(image):
    """QA-mask clouds, snow, shadow and scale NDVI to float."""
    qa = image.select('SummaryQA')
    qa_mask = qa.lte(1)
    dqa = image.select('DetailedQA')
    mask = qa_mask \
        .And(dqa.bitwiseAnd(1 << 10).eq(0)) \
        .And(dqa.bitwiseAnd(1 << 14).eq(0)) \
        .And(dqa.bitwiseAnd(1 << 15).eq(0))
    ndvi = image.select('NDVI').multiply(0.0001).rename('NDVI')
    return ndvi.updateMask(mask) \
        .set('system:time_start', image.get('system:time_start'))


def scale_only(image):
    """Scale NDVI to -1..1 range, no QA masking."""
    ndvi = image.select('NDVI').multiply(0.0001).rename('NDVI')
    return ndvi.set('system:time_start', image.get('system:time_start'))


def garonna_smooth(collection, roi):
    """
    Garonna et al. 2009 smoothing with change reporting.

    Args:
        collection: ee.ImageCollection of NDVI images.
        roi: ee.Geometry for the reduceRegion call.

    Returns dict: {'collection': smoothed IC, 'total_changed_pixels': ee.Dictionary}
    """
    img_list = collection.toList(collection.size())
    n = collection.size()
    indices = ee.List.sequence(0, n.subtract(1))

    def _ndvi(i):
        return ee.Image(img_list.get(
            ee.Number(i).max(0).min(n.subtract(1))
        )).select('NDVI')

    def _flag(boolean_num):
        return ee.Image.constant(
            ee.Number(ee.Algorithms.If(boolean_num, 1, 0)))

    def _smooth(index):
        index = ee.Number(index)
        cur = ee.Image(img_list.get(index))
        v  = cur.select('NDVI')
        p1 = _ndvi(index.subtract(1))
        n1 = _ndvi(index.add(1))
        p2 = _ndvi(index.subtract(2))
        n2 = _ndvi(index.add(2))
        dp = v.subtract(p1)
        dn = v.subtract(n1)
        is_single = (dp.gte(0.3).And(dn.gte(0.3))) \
                 .Or(dp.lte(-0.3).And(dn.lte(-0.3)))
        is_single = is_single.multiply(
            _flag(index.gt(0).And(index.lt(n.subtract(1)))))
        is_pair1 = v.subtract(p1).abs().gte(0.3) \
            .And(n1.subtract(v).abs().lt(0.3)) \
            .And(n2.subtract(n1).abs().gte(0.3))
        is_pair1 = is_pair1.multiply(
            _flag(index.gt(0).And(index.lt(n.subtract(2)))))
        is_pair2 = p1.subtract(p2).abs().gte(0.3) \
            .And(v.subtract(p1).abs().lt(0.3)) \
            .And(n1.subtract(v).abs().gte(0.3))
        is_pair2 = is_pair2.multiply(
            _flag(index.gt(1).And(index.lt(n.subtract(1)))))
        any_change = is_single.Or(is_pair1).Or(is_pair2).rename('changed')
        smoothed = v
        smoothed = smoothed.where(is_single, p1.add(n1).divide(2))
        smoothed = smoothed.where(is_pair1,  p1.add(n2).divide(2))
        smoothed = smoothed.where(is_pair2,  p2.add(n1).divide(2))
        out = cur.addBands(smoothed.rename('NDVI'), overwrite=True)
        return out.addBands(any_change)

    smoothed_col = ee.ImageCollection(indices.map(_smooth))
    total_changed = smoothed_col.select('changed') \
        .sum().reduceRegion(
            reducer=ee.Reducer.sum(), geometry=roi,
            scale=250, maxPixels=1e9)
    clean_col = smoothed_col.select('NDVI')
    return {'collection': clean_col, 'total_changed_pixels': total_changed}


def compute_annual_metrics(collection, start, end, periods_per_year):
    """INDVI (integral proxy = mean * periods), minNDVI, maxNDVI per year."""
    years = ee.List.sequence(start, end)
    def _year(y):
        y = ee.Number(y)
        yc = collection.filter(ee.Filter.calendarRange(y, y, 'year'))
        return ee.Image.cat(
            yc.mean().multiply(periods_per_year).rename('INDVI'),
            yc.min().rename('minNDVI'),
            yc.max().rename('maxNDVI')
        ).set('year', y,
              'system:time_start', ee.Date.fromYMD(y, 1, 1).millis())
    return ee.ImageCollection(years.map(_year))


# ═══════════════════════════════════════════════════════
#  Dynamic World Masking
# ═══════════════════════════════════════════════════════

def build_dw_mask(site_geom, dw_start, dw_end, threshold=0.75):
    """
    Build a binary mask from Dynamic World where undesirable land-cover
    classes (water, crops, built, bare, snow_and_ice) have combined
    probability > threshold.

    Returns an ee.Image: 1 = undesirable (to be masked), 0 = keep.
    """
    all_classes = ['water', 'trees', 'grass', 'flooded_vegetation',
                   'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']
    undesirable = ['water', 'crops', 'built', 'bare', 'snow_and_ice']

    dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
        .filterDate(dw_start, dw_end) \
        .filterBounds(site_geom)

    composite = dw.select(all_classes).median()
    total_prob = composite.reduce(ee.Reducer.sum())
    normalised = composite.divide(total_prob)
    undesirable_sum = normalised.select(undesirable).reduce(ee.Reducer.sum())
    return undesirable_sum.gt(threshold).rename('dw_mask')


# ═══════════════════════════════════════════════════════
#  Mann-Kendall Trend Test
# ═══════════════════════════════════════════════════════

def mk_sign(i, j):
    """
    Pixel-wise sign of (j - i): +1 if j > i, -1 if j < i, 0 if equal.
    Both i and j are single-band ee.Images.
    """
    diff = ee.Image(j).subtract(ee.Image(i))
    combined_mask = ee.Image(i).mask().And(ee.Image(j).mask())
    zero = ee.Image(0).updateMask(combined_mask)
    return zero.where(diff.gt(0), 1).where(diff.lt(0), -1).int16().rename('sign')


def mk_statistic(annual_col):
    """
    Compute pixel-wise Mann-Kendall S statistic.
    annual_col must be a single-band ee.ImageCollection with 'year' property.
    S = sum of sign(x_j - x_i) for all i < j.
    """
    after_filter = ee.Filter.lessThan(
        leftField='year', rightField='year')
    joined = ee.ImageCollection(
        ee.Join.saveAll('after').apply(
            primary=annual_col,
            secondary=annual_col,
            condition=after_filter))

    def _signs_for(current):
        after_coll = ee.ImageCollection.fromImages(current.get('after'))
        return after_coll.map(lambda img: mk_sign(current, img))

    return ee.ImageCollection(
        joined.map(_signs_for).flatten()
    ).reduce('sum', 2).rename('MK_S')


def mk_tie_variance_correction(annual_col):
    """
    Compute the tie-correction term for Kendall's variance.
    Returns sum of t*(t-1)*(2t+5) over all tied groups (per pixel).
    Uses array-based run-length detection (Sen 1968, eq 2.6).
    """
    def _mark_ties(i):
        matches = annual_col.map(lambda j: ee.Image(i).eq(j)).sum()
        return ee.Image(i).multiply(matches.gt(1))
    groups = annual_col.map(_mark_ties)

    arr = groups.toArray()
    length = arr.arrayLength(0)
    indices = ee.Image([1]) \
        .arrayRepeat(0, length) \
        .arrayAccum(0, ee.Reducer.sum()) \
        .toArray(1)
    sorted_arr = arr.arraySort()
    left = sorted_arr.arraySlice(0, 1)
    right = sorted_arr.arraySlice(0, 0, -1)
    mask = left.neq(right) \
        .arrayCat(ee.Image(ee.Array([[1]])), 0)
    run_indices = indices.arrayMask(mask)
    group_sizes = run_indices.arraySlice(0, 1) \
        .subtract(run_indices.arraySlice(0, 0, -1))

    group_factors = group_sizes.expression('b() * (b() - 1) * (b() * 2 + 5)')
    return group_factors.arrayReduce('sum', [0]).arrayGet([0, 0])


def mk_variance(annual_col):
    """
    Kendall's variance: [n(n-1)(2n+5) - sum(t(t-1)(2t+5))] / 18
    where t = size of each tied group.
    """
    n = annual_col.size()
    n_img = ee.Image(n)
    base = n_img.expression('b() * (b() - 1) * (b() * 2 + 5)')
    tie_corr = mk_tie_variance_correction(annual_col)
    return base.subtract(tie_corr).divide(18).float()


def mk_z_score(kendall_s, variance):
    """
    Continuity-corrected Z-score for the Mann-Kendall test.
    Z = (S-1)/sqrt(V) if S>0, (S+1)/sqrt(V) if S<0, 0 if S=0.
    """
    zero = ee.Image(0).updateMask(kendall_s.mask())
    z = zero.where(kendall_s.gt(0), kendall_s.subtract(1).divide(variance.sqrt())) \
            .where(kendall_s.lt(0), kendall_s.add(1).divide(variance.sqrt()))
    return z.rename('MK_S')


def ee_cdf(z):
    """Standard normal CDF via erf: Phi(z) = 0.5 * (1 + erf(z / sqrt(2)))."""
    return ee.Image(0.5).multiply(
        ee.Image(1).add(
            ee.Image(z).divide(ee.Image(2).sqrt()).erf()))


def mk_p_value(z):
    """Two-sided p-value: p = 2 * (1 - Phi(|z|))."""
    return ee.Image(1).subtract(ee_cdf(z.abs())).multiply(2)


def sens_slope(annual_col):
    """
    Sen's slope estimator: median of (x_j - x_i) / (year_j - year_i) for all i < j.
    annual_col must be single-band with 'year' property.
    """
    after_filter = ee.Filter.lessThan(
        leftField='year', rightField='year')
    joined = ee.ImageCollection(
        ee.Join.saveAll('after').apply(
            primary=annual_col,
            secondary=annual_col,
            condition=after_filter))

    def _slopes_for(current):
        after_coll = ee.ImageCollection.fromImages(current.get('after'))
        def _slope(j):
            return ee.Image(j).subtract(current) \
                .divide(ee.Image(j).date().difference(
                    ee.Image(current).date(), 'year')) \
                .rename('slope').float()
        return after_coll.map(_slope)

    slope_images = ee.ImageCollection(joined.map(_slopes_for).flatten())
    return slope_images.reduce(ee.Reducer.median()).rename('Sen_Slope')


# ═══════════════════════════════════════════════════════
#  Export Helpers
# ═══════════════════════════════════════════════════════

def stack_mk(mk_dict, geometry, nodata=-9999):
    """
    Stack Mann-Kendall results into a single multi-band Float32 image.
    Produces 9 bands: MK_S, PValue, SenSlope for each of METRICS.
    """
    bands = []
    for metric in METRICS:
        r = mk_dict[metric]
        bands.extend([
            r['S'].toFloat().rename(f'MK_S_{metric}'),
            r['p'].toFloat().rename(f'PValue_{metric}'),
            r['slope'].toFloat().rename(f'SenSlope_{metric}')
        ])
    return ee.Image.cat(bands).unmask(nodata).clip(geometry)


## 3. Unit Tests
(Essential for agentic programming)


In [3]:
def run_all_tests():
    pt = ee.Geometry.Point([0, 0])

    # ── Helper: make synthetic single-band collection with 'year' ──
    def make_annual_synth(values, start_year=2000):
        """List of floats → single-band ImageCollection with 'year' property."""
        vals = ee.List(values)
        def _make(i):
            i = ee.Number(i)
            y = ee.Number(start_year).add(i)
            return ee.Image.constant(vals.get(i)).toFloat().rename('NDVI') \
                .set('year', y,
                     'system:time_start', ee.Date.fromYMD(y, 1, 1).millis())
        return ee.ImageCollection(
            ee.List.sequence(0, len(values) - 1).map(_make))

    def sample_band(image, band='NDVI', point=pt):
        return image.select(band).sample(point, 1000).first().get(band).getInfo()

    # ═══════ mk_sign ═══════
    a = ee.Image.constant(0.5).toFloat().rename('NDVI')
    b = ee.Image.constant(0.8).toFloat().rename('NDVI')
    c = ee.Image.constant(0.5).toFloat().rename('NDVI')
    assert mk_sign(a, b).sample(pt, 1000).first().get('sign').getInfo() == 1
    assert mk_sign(b, a).sample(pt, 1000).first().get('sign').getInfo() == -1
    assert mk_sign(a, c).sample(pt, 1000).first().get('sign').getInfo() == 0
    print('PASS 1  — mk_sign: +1, -1, 0 correct')

    # ═══════ mk_statistic: monotonic increase ═══════
    mono_up = make_annual_synth([0.1, 0.2, 0.3, 0.4, 0.5])
    s_up = mk_statistic(mono_up)
    s_val = s_up.sample(pt, 1000).first().get('MK_S').getInfo()
    assert s_val == 10, f'FAIL mk_statistic mono up: expected 10, got {s_val}'
    print(f'PASS 2  — mk_statistic monotonic increase: S = {s_val}')

    # ═══════ mk_statistic: monotonic decrease ═══════
    mono_dn = make_annual_synth([0.5, 0.4, 0.3, 0.2, 0.1])
    s_dn = mk_statistic(mono_dn).sample(pt, 1000).first().get('MK_S').getInfo()
    assert s_dn == -10, f'FAIL mk_statistic mono down: expected -10, got {s_dn}'
    print(f'PASS 3  — mk_statistic monotonic decrease: S = {s_dn}')

    # ═══════ mk_statistic: flat ═══════
    flat = make_annual_synth([0.5, 0.5, 0.5, 0.5, 0.5])
    s_flat = mk_statistic(flat).sample(pt, 1000).first().get('MK_S').getInfo()
    assert s_flat == 0, f'FAIL mk_statistic flat: expected 0, got {s_flat}'
    print(f'PASS 4  — mk_statistic flat: S = {s_flat}')

    # ═══════ mk_variance: no ties ═══════
    # n=5, no ties: V = 5*4*15 / 18 = 300/18 = 16.667
    var_no_ties = mk_variance(mono_up)
    v_val = var_no_ties.sample(pt, 1000).first().get('constant').getInfo()
    expected_v = 5 * 4 * 15 / 18
    assert abs(v_val - expected_v) < 0.1, f'FAIL variance no ties: {v_val} != {expected_v}'
    print(f'PASS 5  — mk_variance (no ties): V = {v_val:.2f} (expected {expected_v:.2f})')

    # ═══════ mk_z_score ═══════
    z = mk_z_score(s_up, var_no_ties)
    z_val = z.sample(pt, 1000).first().get('MK_S').getInfo()
    expected_z = (10 - 1) / (expected_v ** 0.5)
    assert abs(z_val - expected_z) < 0.1, f'FAIL z-score: {z_val} != {expected_z}'
    print(f'PASS 6  — mk_z_score: Z = {z_val:.3f} (expected {expected_z:.3f})')

    # ═══════ mk_p_value ═══════
    p = mk_p_value(z)
    p_val = p.sample(pt, 1000).first().get('constant').getInfo()
    assert 0 < p_val < 0.05, f'FAIL p-value: {p_val} not < 0.05 for strong trend'
    print(f'PASS 7  — mk_p_value: p = {p_val:.4f} (< 0.05 for strong monotonic trend)')

    # ═══════ ee_cdf: Z=0 → 0.5 ═══════
    cdf_0 = ee_cdf(ee.Image(0)).sample(pt, 1000).first().get('constant').getInfo()
    assert abs(cdf_0 - 0.5) < 0.01, f'FAIL cdf(0): {cdf_0}'
    print(f'PASS 8  — ee_cdf(0) = {cdf_0:.4f}')

    # ═══════ sens_slope: linear increase ═══════
    # 0.1 per year increase: slope should be 0.1
    slope_img = sens_slope(mono_up)
    slope_val = slope_img.sample(pt, 1000).first().get('Sen_Slope').getInfo()
    assert abs(slope_val - 0.1) < 0.02, f'FAIL sens_slope: {slope_val} != 0.1'
    print(f'PASS 9  — sens_slope (linear): slope = {slope_val:.4f} (expected 0.1)')

    # ═══════ sens_slope: flat → 0 ═══════
    slope_flat = sens_slope(flat).sample(pt, 1000).first().get('Sen_Slope').getInfo()
    assert abs(slope_flat) < 0.01, f'FAIL sens_slope flat: {slope_flat}'
    print(f'PASS 10 — sens_slope (flat): slope = {slope_flat:.4f}')

    # ═══════ build_dw_mask: returns binary ═══════
    test_geom = ee.Geometry.Point([-0.36, 50.96]).buffer(1000).bounds()
    mask = build_dw_mask(test_geom, '2022-04-01', '2022-09-30', 0.75)
    bands = mask.bandNames().getInfo()
    assert bands == ['dw_mask'], f'FAIL dw_mask bands: {bands}'
    stats = mask.reduceRegion(
        reducer=ee.Reducer.minMax(), geometry=test_geom,
        scale=250, maxPixels=1e9).getInfo()
    assert stats['dw_mask_min'] in [0, 1] and stats['dw_mask_max'] in [0, 1], \
        f'FAIL dw_mask range: {stats}'
    print(f'PASS 11 — build_dw_mask: bands={bands}, range=[{stats["dw_mask_min"]}, {stats["dw_mask_max"]}]')

    # ═══════ scale_only ═══════
    test_img = ee.Image.constant(5000).rename('NDVI').set('system:time_start', 123)
    scaled = scale_only(test_img)
    scaled_val = scaled.sample(pt, 1).first().get('NDVI').getInfo()
    assert abs(scaled_val - 0.5) < 0.0001, f'FAIL scale_only: {scaled_val} != 0.5'
    print(f'PASS 12 — scale_only: 5000 → {scaled_val}')

    # ═══════ prep_modis: good QA passes, bad QA masks ═══════
    good_qa = ee.Image.constant([5000, 0, 0]).rename(['NDVI', 'SummaryQA', 'DetailedQA']) \
        .set('system:time_start', 1)
    bad_qa = ee.Image.constant([5000, 0, 1024]).rename(['NDVI', 'SummaryQA', 'DetailedQA']) \
        .set('system:time_start', 1)  # bit 10 = cloud shadow
    good_result = prep_modis(good_qa)
    bad_result = prep_modis(bad_qa)
    good_mask_val = good_result.mask().sample(pt, 1).first().get('NDVI').getInfo()
    bad_mask_val = bad_result.mask().sample(pt, 1).first().get('NDVI').getInfo()
    assert good_mask_val == 1, f'FAIL prep_modis good: mask={good_mask_val}'
    assert bad_mask_val == 0, f'FAIL prep_modis bad: mask={bad_mask_val}'
    print(f'PASS 13 — prep_modis: good QA unmasked, cloud-shadow QA masked')

    # ═══════ compute_annual_metrics ═══════
    im1 = ee.Image.constant(0.4).toFloat().rename('NDVI') \
        .set('system:time_start', ee.Date('2000-03-01').millis())
    im2 = ee.Image.constant(0.6).toFloat().rename('NDVI') \
        .set('system:time_start', ee.Date('2000-07-01').millis())
    test_col = ee.ImageCollection([im1, im2])
    annual = compute_annual_metrics(test_col, 2000, 2000, 23)
    annual_img = annual.first()
    indvi = annual_img.sample(pt, 1).first().get('INDVI').getInfo()
    min_ndvi = annual_img.sample(pt, 1).first().get('minNDVI').getInfo()
    max_ndvi = annual_img.sample(pt, 1).first().get('maxNDVI').getInfo()
    expected_indvi = 0.5 * 23  # mean=0.5, periods=23 => 11.5
    assert abs(indvi - expected_indvi) < 0.01, f'FAIL INDVI: {indvi} != {expected_indvi}'
    assert abs(min_ndvi - 0.4) < 0.01, f'FAIL minNDVI: {min_ndvi}'
    assert abs(max_ndvi - 0.6) < 0.01, f'FAIL maxNDVI: {max_ndvi}'
    print(f'PASS 14 — compute_annual_metrics: INDVI={indvi}, min={min_ndvi}, max={max_ndvi}')

    # ═══════ garonna_smooth: smoke test ═══════
    # Verify returns correct structure and preserves collection size
    smoke_col = make_annual_synth([0.2, 0.3, 0.4, 0.3, 0.2])
    smoke_roi = pt.buffer(1000).bounds()
    result = garonna_smooth(smoke_col, roi=smoke_roi)
    assert 'collection' in result, 'FAIL garonna_smooth: missing collection key'
    assert 'total_changed_pixels' in result, 'FAIL garonna_smooth: missing total_changed_pixels key'
    out_size = result['collection'].size().getInfo()
    assert out_size == 5, f'FAIL garonna_smooth: output size {out_size} != 5'
    print(f'PASS 15 — garonna_smooth: returns correct structure, size preserved ({out_size})')

    print('\n✅ All 15 tests passed!')

run_all_tests()


PASS 1  — mk_sign: +1, -1, 0 correct
PASS 2  — mk_statistic monotonic increase: S = 10
PASS 3  — mk_statistic monotonic decrease: S = -10
PASS 4  — mk_statistic flat: S = 0
PASS 5  — mk_variance (no ties): V = 16.67 (expected 16.67)
PASS 6  — mk_z_score: Z = 2.205 (expected 2.205)
PASS 7  — mk_p_value: p = 0.0275 (< 0.05 for strong monotonic trend)
PASS 8  — ee_cdf(0) = 0.5000
PASS 9  — sens_slope (linear): slope = 0.1001 (expected 0.1)
PASS 10 — sens_slope (flat): slope = 0.0000
PASS 11 — build_dw_mask: bands=['dw_mask'], range=[0, 1]
PASS 12 — scale_only: 5000 → 0.5
PASS 13 — prep_modis: good QA unmasked, cloud-shadow QA masked
PASS 14 — compute_annual_metrics: INDVI=11.5, min=0.4000000059604645, max=0.6000000238418579
PASS 15 — garonna_smooth: returns correct structure, size preserved (5)

✅ All 15 tests passed!


## 4. Site Setup & Dynamic World Mask

In [5]:
# Load raw FC and merge features sharing a site_id into single features
raw_fc = ee.FeatureCollection(SITES_ASSET)
site_ids = raw_fc.aggregate_array('site_id').distinct().getInfo()

def merge_site(site_id):
    subset = raw_fc.filter(ee.Filter.eq('site_id', site_id))
    merged_geom = subset.geometry().dissolve(maxError=1)
    return ee.Feature(merged_geom, subset.first().toDictionary())

sites_fc = ee.FeatureCollection([merge_site(s) for s in site_ids])

# Print site metadata and export as CSV for R visualisation
print(f'Sites to process ({len(site_ids)}):')
site_meta = []
for s in site_ids:
    yr = raw_fc.filter(ee.Filter.eq('site_id', s)).first().get('start_year').getInfo()
    print(f'  {s}: {yr}-{END_YEAR}')
    site_meta.append({'site_id': s, 'start_year': yr, 'end_year': END_YEAR})


Sites to process (20):
  Central_Apennines: 2000-2025
  Border_Meuse: 2000-2025
  Greater_Coa_Valley: 2000-2025
  Knepp: 2001-2025
  Oostvaardersplassen: 2000-2025
  Southern_Carpathians: 2011-2025
  Dundreggan: 2008-2025
  Pentezug: 2000-2025
  Alladale: 2000-2025
  Chernobyl_Exclusion_Zone: 2000-2025
  Doberitzer_Heide: 2000-2025
  Swiss_National_Park: 2000-2025
  Tarutino_Steppe: 2004-2025
  Wild_Ennerdale: 2003-2025
  Kempen_Broek: 2010-2025
  Danube_Delta: 2013-2025
  Naliboki_Forest: 2000-2025
  Velebit_Mountains: 2012-2025
  Creag_Meagaidh: 2000-2025
  Vanatori_Neamt_Nature_Park: 2012-2025


## 5. Compute Annual Metrics & Run Mann-Kendall (all sites)


In [ ]:
all_site_results = {}

for site_id in site_ids:
    print(f'\n{"="*60}')
    print(f'  Processing: {site_id}')
    print(f'{"="*60}')

    site = sites_fc.filter(ee.Filter.eq('site_id', site_id))
    geometry = site.geometry().dissolve(maxError=1)
    site_start = ee.Number(site.first().get('start_year')).getInfo()

    # ── Dynamic World mask ─────────────────────────────
    dw_mask = build_dw_mask(geometry, DW_START, DW_END, DW_PROB_THRESHOLD)

    # ── 250m (Garonna smoothed 16-day) ────────────────
    col_250 = ee.ImageCollection('MODIS/061/MOD13Q1') \
        .filterBounds(geometry) \
        .filterDate(f'{site_start}-01-01', f'{END_YEAR}-12-31') \
        .map(lambda img: img.clip(geometry))
    smooth_250 = garonna_smooth(col_250.map(prep_modis), roi=geometry)['collection']
    annual_250 = compute_annual_metrics(smooth_250, site_start, END_YEAR, 23)

    # ── 1000m (raw monthly, no smoothing) ─────────────
    col_1000 = ee.ImageCollection('MODIS/061/MOD13A3') \
        .filterBounds(geometry) \
        .filterDate(f'{site_start}-01-01', f'{END_YEAR}-12-31') \
        .map(lambda img: img.clip(geometry))
    annual_1000 = compute_annual_metrics(col_1000.map(scale_only), site_start, END_YEAR, 12)

    # ── Apply DW mask ──────────────────────────────────
    cleaned_250 = annual_250.map(lambda img: img.updateMask(dw_mask.Not()))
    cleaned_1000 = annual_1000.map(lambda img: img.updateMask(dw_mask.Not()))

    # ── Mann-Kendall ───────────────────────────────────
    mk_results_250 = {}
    mk_results_1000 = {}
    for metric in METRICS:
        # 250m
        mc250 = cleaned_250.select(metric) \
            .map(lambda img: img.rename('NDVI').copyProperties(img, img.propertyNames()))
        s2, v2 = mk_statistic(mc250), mk_variance(mc250)
        z2 = mk_z_score(s2, v2)
        mk_results_250[metric] = {'S': s2, 'variance': v2, 'Z': z2, 'p': mk_p_value(z2), 'slope': sens_slope(mc250)}

        # 1000m
        mc1000 = cleaned_1000.select(metric) \
            .map(lambda img: img.rename('NDVI').copyProperties(img, img.propertyNames()))
        s1, v1 = mk_statistic(mc1000), mk_variance(mc1000)
        z1 = mk_z_score(s1, v1)
        mk_results_1000[metric] = {'S': s1, 'variance': v1, 'Z': z1, 'p': mk_p_value(z1), 'slope': sens_slope(mc1000)}

    print(f'  MK stats computed for 250m & 1000m.')

    all_site_results[site_id] = {
        'geometry': geometry,
        'mk': mk_results_250,
        'mk_1000': mk_results_1000,
    }

print(f'\n✅ All {len(site_ids)} sites processed')


## 6. Export Results to Drive (all sites)


In [ ]:
nodata = -9999
task_count = 0
for site_id, res in all_site_results.items():
    geometry = res['geometry']

    # Export 250m
    ee.batch.Export.image.toDrive(
        image=stack_mk(res['mk'], geometry, nodata),
        description=f'MK_Stacked_250m_{site_id}',
        folder='GEE_MK_250m',
        fileNamePrefix=f'MK_Stacked_250m_{site_id}',
        region=geometry, scale=250, maxPixels=1e13,
        formatOptions={'noData': nodata}).start()

    # Export 1000m
    ee.batch.Export.image.toDrive(
        image=stack_mk(res['mk_1000'], geometry, nodata),
        description=f'MK_Stacked_1000m_{site_id}',
        folder='GEE_MK_1000m',
        fileNamePrefix=f'MK_Stacked_1000m_{site_id}',
        region=geometry, scale=1000, maxPixels=1e13,
        formatOptions={'noData': nodata}).start()

    task_count += 2
    print(f'Exports started for {site_id} (250m & 1000m)')

print(f'\n✅ {task_count} export tasks submitted — check GEE Tasks tab')
